# Background Tasks and Streaming

This notebook covers:

1. Schedule work to run **after** the response is sent, with `BackgroundTasks`
2. Stream large or open-ended responses with `StreamingResponse`
3. Push live updates to the client via **Server-Sent Events** (SSE)
4. Recognize when to graduate from in-process background work to a real queue (Celery / RQ / Arq / SQS)

**Scope**: FastAPI + `httpx.AsyncClient` against the in-process app. The portfolio/asset domain reappears: post-create notifications, CSV export of holdings, live price ticks.

Notebooks 3.1 and 3.2 were about *running* a single request without blocking the loop. This one is about request shapes where the client either doesn't need to wait (background work) or needs to receive data progressively (streaming, SSE).

## 1. Why Background Work

Some endpoints do two things:

1. **The thing the client needs an answer for.** "Did you create the order? What's the ID?"
2. **The thing the system needs to do *because* of it.** Send a confirmation email, append to an audit log, push a webhook, invalidate a cache.

The client should not wait on (2). It often doesn't even need to know (2) succeeded synchronously. The pattern is: return the response immediately, run (2) afterwards.

FastAPI's `BackgroundTasks` is the lightest tool for this. It runs the deferred work **in the same worker process**, *after* the response is flushed to the client. No external dependency, no queue, no broker. The catch (covered in §5) is that "same process" also means "lost on crash / restart" — fine for fire-and-forget side effects, not fine for anything you need durability guarantees on.

## 2. `BackgroundTasks`

Inject a `BackgroundTasks` parameter and call `.add_task(fn, *args, **kwargs)`. FastAPI runs the function after the response is sent — sync functions go to the threadpool, async functions go on the loop, same rules as request handlers (notebook 3.1).

In [ ]:
import asyncio, time
from fastapi import FastAPI, BackgroundTasks
from httpx import AsyncClient, ASGITransport

app = FastAPI()

# Stand-in for an audit / notification side effect. In real code this would be
# a database write, an email send, or an HTTP push.
NOTIFIED: list[dict] = []

async def notify_async(asset_id: int, ticker: str):
    await asyncio.sleep(0.05)  # pretend HTTP call to notification service
    NOTIFIED.append({"asset_id": asset_id, "ticker": ticker, "at": time.time()})

@app.post("/assets", status_code=201)
async def create_asset(ticker: str, bg: BackgroundTasks):
    # Imagine we inserted into the DB and got an id back.
    asset_id = len(NOTIFIED) + 1
    bg.add_task(notify_async, asset_id, ticker)
    # The response goes out NOW. notify_async runs after.
    return {"id": asset_id, "ticker": ticker}

async def demo():
    async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as ac:
        r = await ac.post("/assets", params={"ticker": "AAPL"})
    print(f"response: {r.json()}")
    # By the time the in-process test client returns, the background task has typically
    # already finished — but the handler never awaited it. NOTIFIED was populated
    # *after* the response object was constructed.
    print(f"notified after the fact: {NOTIFIED}")

asyncio.run(demo())

**A measurement caveat first.** Over a real network connection, the client receives the response bytes the moment the server finishes writing them, and the background task runs after that. With `ASGITransport` (in-process), the client `await` only resolves after the full ASGI request cycle — including the background task — completes. So you can't time the "client got response first" effect from inside this notebook. The lesson holds in production; the test transport collapses it.

A few rules to internalize:

- **Background tasks run after the response is sent**, but **before the worker is fully idle**. A crash or `kill -9` between response-send and task-completion drops the task. There is no retry.
- **An exception inside a background task does not affect the response** the client already saw. It will be logged but not surfaced. Use a real handler if you need to fail loudly.
- **They share resources with the worker**: the same DB connections, the same memory. A runaway background task can starve later requests.

Use `BackgroundTasks` for tasks where "we tried, it's logged, we move on" is acceptable. Anything stronger — guaranteed delivery, retry, scheduling, fan-out across workers — wants a real queue (§5).

## 3. `StreamingResponse`

The default `JSONResponse` buffers the whole body in memory before sending. For a 10MB CSV export of every holding in every portfolio, that's wasteful: the server holds the full body while the client waits.

`StreamingResponse` takes any async or sync iterable and pipes chunks to the client as they're produced. The server's memory stays flat regardless of total response size, and the client starts receiving data sooner.

In [ ]:
from fastapi.responses import StreamingResponse

# A pretend large dataset: 10k holdings across portfolios.
HOLDINGS = [
    {"portfolio_id": p, "ticker": f"T{i:04d}", "qty": i * 10, "price": 100 + i * 0.01}
    for p in range(1, 6)
    for i in range(2_000)
]

async def holdings_csv_rows():
    # Header
    yield "portfolio_id,ticker,qty,price\n"
    # Stream in 500-row chunks so we batch the network writes.
    chunk = []
    for h in HOLDINGS:
        chunk.append(f"{h['portfolio_id']},{h['ticker']},{h['qty']},{h['price']}\n")
        if len(chunk) >= 500:
            yield "".join(chunk)
            chunk = []
            await asyncio.sleep(0)  # cooperatively yield to the loop between chunks
    if chunk:
        yield "".join(chunk)

@app.get("/holdings.csv")
async def holdings_csv():
    return StreamingResponse(
        holdings_csv_rows(),
        media_type="text/csv",
        headers={"Content-Disposition": 'attachment; filename="holdings.csv"'},
    )

async def demo():
    async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as ac:
        # Stream the response and count bytes/rows as they arrive.
        async with ac.stream("GET", "/holdings.csv") as r:
            n_bytes = 0
            n_rows = 0
            async for chunk in r.aiter_bytes():
                n_bytes += len(chunk)
                n_rows += chunk.count(b"\n")
            print(f"received {n_bytes} bytes, {n_rows} rows (incl. header), content-type={r.headers['content-type']}")

asyncio.run(demo())

Things to notice:

- **No `list(...)`, no big string concatenation** on the server. The CSV is produced row-by-row and flushed in chunks. Memory is `O(chunk_size)`, not `O(total_rows)`.
- **The async generator can `await`** between rows. In a real export it might fetch each page from the DB asynchronously while the prior page is being sent.
- **The `Content-Disposition` header** turns a CSV stream into a file download in the browser. Without it the browser would try to render the bytes inline.
- The same pattern works for any large response: JSON Lines (`application/x-ndjson`), log tails, generated reports, even media.

## 4. Server-Sent Events (SSE)

SSE is a special case of streaming: long-lived, one-way push from server to client over plain HTTP, with a well-known wire format the browser understands natively (`EventSource` in JS).

The format is text:

```
data: {"price": 100.0}\n
\n
data: {"price": 100.2}\n
\n
```

Each event is `data: <payload>\n\n`. The double newline marks event boundaries. The `Content-Type` is `text/event-stream`. You can also send `event: <name>` and `id: <id>` lines for typed events and reconnect support.

When to reach for SSE instead of WebSockets:

- **One-way push** (server → client): SSE is simpler, no library on the client, auto-reconnect built in.
- **Bidirectional** or binary payloads: WebSockets.

We'll demonstrate with a fake live-price stream — exactly the kind of thing a "watchlist" UI subscribes to.

In [ ]:
import json, random

random.seed(0)

async def price_event_stream(ticker: str, n_ticks: int = 5):
    price = 100.0
    for i in range(n_ticks):
        price += random.uniform(-0.5, 0.5)
        payload = {"ticker": ticker, "tick": i, "price": round(price, 2)}
        yield f"data: {json.dumps(payload)}\n\n"
        await asyncio.sleep(0.02)  # 50 Hz tick rate

@app.get("/prices/{ticker}/stream")
async def prices_stream(ticker: str):
    return StreamingResponse(
        price_event_stream(ticker),
        media_type="text/event-stream",
    )

async def demo():
    async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as ac:
        async with ac.stream("GET", "/prices/AAPL/stream") as r:
            print(f"content-type: {r.headers['content-type']}")
            async for line in r.aiter_lines():
                if line.startswith("data:"):
                    print("event:", line[len("data: "):])

asyncio.run(demo())

On the browser side, this same endpoint would be consumed with two lines of JavaScript:

```javascript
const es = new EventSource("/prices/AAPL/stream");
es.onmessage = (e) => console.log(JSON.parse(e.data));
```

`EventSource` handles reconnection automatically. If the connection drops, the browser retries with an exponential backoff, and if the server included `id:` lines, it sends a `Last-Event-ID` header so you can resume from where the client left off.

A few practical notes:

- **SSE requires HTTP/1.1 or HTTP/2 with keep-alive.** Most reverse proxies pass it through fine, but watch for buffering: nginx by default buffers responses and will batch your events. Set `X-Accel-Buffering: no` (header) or `proxy_buffering off` (config) to disable.
- **The connection stays open.** A single uvicorn worker holding 1000 SSE connections is bound by file descriptors, not CPU. But it's also a connection slot a load balancer can't reclaim — plan capacity accordingly.
- **`await asyncio.sleep(0)` between events** is enough to yield the loop. You do not need to throttle aggressively; the OS write buffer back-pressures you naturally.

## 5. When You Need a Real Queue

`BackgroundTasks` and in-process streaming both share one weakness: they live and die with the worker. As soon as you need any of the following, graduate to a real queue:

| You want…                                          | Tool                                          |
|----------------------------------------------------|-----------------------------------------------|
| Durability across restarts                         | Celery, RQ, Arq, SQS — anything with a broker |
| Automatic retries on failure                       | Same — pick one with retry policies built in  |
| Work that takes minutes or longer                  | Same. Don't tie up a request slot.            |
| Scheduling / cron-style triggers                   | Celery beat, APScheduler, cloud schedulers    |
| Fan-out across multiple worker pools               | Real queue                                    |
| Observability: "which jobs ran, which failed, when" | Real queue (most ship with a dashboard)       |

The architectural shape changes:

- The handler no longer *does* the work. It **enqueues a job** (a row in Redis, SQS, Postgres) and returns immediately with a job ID.
- A separate worker process pulls jobs and runs them. Crashes are tolerated by the broker — the job stays in the queue.
- The client polls a status endpoint (`GET /jobs/{id}`) or subscribes via SSE/WebSocket for completion. We have all the pieces to build either in this chapter.

`BackgroundTasks` is the right tool for *side effects that don't matter much*. The moment they matter, get a queue.

We won't deploy a broker in this notebook — the choice (Redis-backed Celery vs Postgres-backed Arq vs cloud SQS) is environment-specific. But the *handler-side* shape is identical to what we've already built: an endpoint that hands work off and gets out of the way.

## Key Takeaways

- **`BackgroundTasks`** runs work after the response is sent, in the same worker process. Great for fire-and-forget side effects (notify, audit). No durability, no retries.
- **`StreamingResponse`** yields chunks as they're produced. Keeps server memory `O(chunk)` for arbitrarily large bodies. Pair with `Content-Disposition` for file downloads.
- **Server-Sent Events** are streaming + `text/event-stream` + the `data: ...\n\n` format. One-way push, simpler than WebSockets, native browser support.
- **The bigger pattern**: handlers should hand off slow or unreliable work, then get out of the way. The variants are just *to whom* — a background task, the response stream itself, or an external queue.
- **Capstone tie-in**: the portfolio API will expose a CSV export of holdings (`StreamingResponse`), a live-prices feed for the watchlist (SSE), and a post-create webhook trigger (`BackgroundTasks` to start, queue-backed once it matters).

## Exercises

**1. SSE counter.** Build a `GET /count?n=10` endpoint that emits `data: {"i": k}\n\n` for `k in range(n)` with a 100ms pause between each. Consume it via `httpx.AsyncClient` and confirm the 10 events arrive over ~1 second.

**2. Background task with failure.** Add a `BackgroundTasks` job that *raises* halfway through (e.g., `raise ValueError("boom")`). What does the client see? What appears in logs? Now wrap the task body in `try / except` and log instead — explain why you'd choose this pattern over letting it raise.

**3. Streaming JSONL export.** Convert the `/holdings.csv` endpoint to a `/holdings.jsonl` endpoint that streams one JSON object per line (`application/x-ndjson`). Each line should be a `json.dumps(h)` for one holding. Confirm the response can be parsed line-by-line on the client and that memory stays flat for a large dataset.